# 허깅페이스의 임베딩 모델
허깅페이스에는 여러 임베딩 모델이 존재합니다. 이 중 한글을 가장 잘 임베딩 할 수 있는 모델인 `bge-m3` 모델을 사용해 문장을 임베딩하고, 유사도를 구해봅니다.

In [9]:
!pip install -U sentence-transformers

# Python 라이브러리로, 자연어 처리(NLP) 작업에서 문장이나 텍스트의 의미를 효율적으로 벡터로 변환할 수 있도록 함. 
# 이를 통해 문장의 의미를 숫자 형태로 표현할 수 있으며, 이 벡터 표현을 사용하여 텍스트 간의 유사도 측정, 검색, 분류, 군집화 등 다양한 NLP 작업을 수행.

`BAAI/bge-m3`의 스펙 정보는 huggingface에서 확인이 가능합니다.

https://huggingface.co/BAAI/bge-m3/blob/main/config.json

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # 또는 "BAAI/bge-m3"
print("Model loaded successfully!")

/Users/khb43/anaconda3/envs/dl_lecture-env/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Model loaded successfully!


## bge-m3 임베딩 수행

In [3]:
embedded_vector = model.encode("야 저기 차 온다")
embedded_vector.shape

(384,)

In [4]:
import pandas as pd

data = [
    "내일 차타고 놀러가자",
    "지금 오는 버스는 어디서 오는 버스야?",
    "차 한잔 하면서 이야기 하시죠",
    "5차 공동구매! 오늘만 세일!",
    "홍차 녹차 중에 어떤 차가 좋으세요?",
]

df = pd.DataFrame(data, columns=['text'])
df

,text
0,내일 차타고 놀러가자
1,지금 오는 버스는 어디서 오는 버스야?
2,차 한잔 하면서 이야기 하시죠
3,5차 공동구매! 오늘만 세일!
4,홍차 녹차 중에 어떤 차가 좋으세요?


In [5]:
def get_embedding(text):
    return list(model.encode(text))

df['embedding'] = df.apply(lambda row : get_embedding(row.text), axis=1)
# list Comprehension?
# df['embedding'] = [get_embedding(text) for text in df['text']]

df

,text,embedding
0,내일 차타고 놀러가자,"[-0.012961757, 0.09570865, 0.012395903, -0.049..."
1,지금 오는 버스는 어디서 오는 버스야?,"[-0.017100615, 0.051439293, 0.07252197, -0.033..."
2,차 한잔 하면서 이야기 하시죠,"[-0.03483857, 0.06232402, 0.00910084, -0.02110..."
3,5차 공동구매! 오늘만 세일!,"[-0.035357684, 0.08208709, 0.09317632, -0.0349..."
4,홍차 녹차 중에 어떤 차가 좋으세요?,"[-0.05502164, 0.1100795, 0.037254743, -0.00350..."


# 유사도 구하기
- 벡터의 유사도를 구하는 방법 중 가장 쉽고 간편한 방법인 코사인 유사도 구현

In [6]:
import numpy as np

def cos_sim(A, B):
  return A @ B/(np.linalg.norm(A)*np.linalg.norm(B))

In [7]:
def return_answer_candidate(df, query):
    '''
        df : 문장과 임베딩 벡터가 들어있는 데이터 프레임
        query : 질의할 문장
    '''
    query_embedding = get_embedding( query )
    df["similarity"] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))
    # list Comprehension?
    # df["similarity"] = [cos_sim(np.array(x), np.array(query_embedding)) for x in df.embedding]


    # 입력한 문장(query)와 데이터 세트 내에 있는 문장의 유사도를 내림차순으로 정렬
    results_co = df.sort_values("similarity", ascending=False, ignore_index=True)
    return results_co.head(3) # 유사도가 가장 비슷한 top 3를 리턴

In [8]:
sim_result = return_answer_candidate(df, "야 저기 차 온다")
sim_result

,text,embedding,similarity
0,차 한잔 하면서 이야기 하시죠,"[-0.03483857, 0.06232402, 0.00910084, -0.02110...",0.795987
1,홍차 녹차 중에 어떤 차가 좋으세요?,"[-0.05502164, 0.1100795, 0.037254743, -0.00350...",0.791277
2,내일 차타고 놀러가자,"[-0.012961757, 0.09570865, 0.012395903, -0.049...",0.769974


In [9]:
sim_result = return_answer_candidate(df, "예쁜 카페 가고 싶어")
sim_result

,text,embedding,similarity
0,홍차 녹차 중에 어떤 차가 좋으세요?,"[-0.05502164, 0.1100795, 0.037254743, -0.00350...",0.643708
1,내일 차타고 놀러가자,"[-0.012961757, 0.09570865, 0.012395903, -0.049...",0.640283
2,5차 공동구매! 오늘만 세일!,"[-0.035357684, 0.08208709, 0.09317632, -0.0349...",0.629427


In [10]:
sim_result = return_answer_candidate(df, "3차 특가 세일! 오늘이 기회")
sim_result

,text,embedding,similarity
0,5차 공동구매! 오늘만 세일!,"[-0.035357684, 0.08208709, 0.09317632, -0.0349...",0.770247
1,홍차 녹차 중에 어떤 차가 좋으세요?,"[-0.05502164, 0.1100795, 0.037254743, -0.00350...",0.712147
2,내일 차타고 놀러가자,"[-0.012961757, 0.09570865, 0.012395903, -0.049...",0.708452
